In [1]:
import requests

In [2]:
request = requests.get("https://www.basketball-reference.com/teams/MIA/2026.html")
print(request.status_code)

403


In [7]:
import pandas as pd

def get_team_players(url: str):
    # Read all tables on the page
    tables = pd.read_html(url)

    roster_df = None

    # Find the roster table (the one with a 'Player' column and reasonable size)
    for df in tables:
        if 'Player' in df.columns and len(df) <= 30:
            roster_df = df
            break

    if roster_df is None:
        raise ValueError("Could not find roster table with a 'Player' column.")

    # Get unique player names as a list
    players = roster_df['Player'].dropna().unique().tolist()
    return roster_df

if __name__ == "__main__":
    temp_team = "MIA"
    url = f"https://www.basketball-reference.com/teams/{temp_team}/2026.html"
    players = get_team_players(url)

    for name in players:
        print(name)


No.
Player
Pos
Ht
Wt
Birth Date
Birth
Exp
College


In [10]:
roster_df = get_team_players(url)
roster_df.head()

,No.,Player,Pos,Ht,Wt,Birth Date,Birth,Exp,College
0,7.0,Kel'el Ware,C,7-0,230,"April 20, 2004",us US,1,"Oregon, Indiana"
1,12.0,Dru Smith,SG,6-2,203,"December 30, 1997",us US,3,"University of Evansville, Missouri"
2,11.0,Jaime Jaquez Jr.,SF,6-6,225,"February 18, 2001",us US,2,UCLA
3,45.0,Davion Mitchell,PG,6-0,202,"September 5, 1998",us US,4,"Auburn, Baylor"
4,0.0,Simone Fontecchio,SF,6-7,209,"December 9, 1995",it IT,3,NaN


In [32]:
from selenium import webdriver
from bs4 import BeautifulSoup, Comment
from selenium.webdriver.chrome.service import Service

url = "https://www.basketball-reference.com/teams/MIA/2026.html"

# Start Chrome
driver = webdriver.Chrome()
driver.get(url)

html = driver.page_source
driver.quit()

soup = BeautifulSoup(html, "html.parser")

# Basketball-reference hides tables in comments sometimes


In [44]:
tables = soup.find("table", {"id": "roster"})
tables

<table class="sortable stats_table now_sortable" data-cols-to-freeze=",2" id="roster">
<caption>Roster Table</caption>
<colgroup><col/><col/><col/><col/><col/><col/><col/><col/><col/></colgroup>
<thead>
<tr>
<th aria-label="No." class="poptip sort_default_asc center" data-stat="number" data-tip="Uniform Number" scope="col">No.</th>
<th aria-label="Player" class="poptip sort_default_asc center" data-stat="player" scope="col">Player</th>
<th aria-label="Pos" class="poptip sort_default_asc center" data-stat="pos" data-tip="Position" scope="col">Pos</th>
<th aria-label="Ht" class="poptip sort_default_asc center" data-stat="height" data-tip="Height" scope="col">Ht</th>
<th aria-label="Wt" class="poptip sort_default_asc center" data-stat="weight" data-tip="Weight" scope="col">Wt</th>
<th aria-label="Birth Date" class="poptip sort_default_asc center" data-stat="birth_date" scope="col">Birth Date</th>
<th aria-label="Birth" class="poptip center" data-stat="flag" data-tip="Country of Birth" sco

In [47]:
rows =tables.find("tbody").find_all("tr")
rows


[<tr data-row="0"><th class="center" data-stat="number" scope="row">7</th><td class="left" csk="Ware,Kel'el" data-stat="player"><a href="/players/w/wareke01.html">Kel'el Ware</a></td><td class="center" csk="5" data-stat="pos">C</td><td class="right" csk="84.0" data-stat="height">7-0</td><td class="right" data-stat="weight">230</td><td class="left" csk="20040420" data-stat="birth_date">April 20, 2004</td><td class="left" data-stat="flag"><span class="f-i f-us" style="">us</span> US</td><td class="right" csk="1" data-stat="years_experience">1</td><td class="left" data-stat="college"><a href="/friv/colleges.fcgi?college=oregon">Oregon</a>, <a href="/friv/colleges.fcgi?college=indiana">Indiana</a></td></tr>,
 <tr data-row="1"><th class="center" data-stat="number" scope="row">12</th><td class="left" csk="Smith,Dru" data-stat="player"><a href="/players/s/smithdr01.html">Dru Smith</a></td><td class="center" csk="2" data-stat="pos">SG</td><td class="right" csk="74.0" data-stat="height">6-2</td

In [49]:
from bs4 import BeautifulSoup

player = {}
row = rows[0]
# row = <your tr bs4 element>

# jersey number
player["number"] = row.find("th", {"data-stat": "number"}).get_text(strip=True)

# player cell
player_cell = row.find("td", {"data-stat": "player"})

player["name"] = player_cell.get_text(strip=True)

# full href
player["href"] = "https://www.basketball-reference.com" + player_cell.find("a")["href"]

# birth date
player["birth_date"] = row.find("td", {"data-stat": "birth_date"}).get_text(strip=True)

print(player)


{'number': '7', 'name': "Kel'el Ware", 'href': 'https://www.basketball-reference.com/players/w/wareke01.html', 'birth_date': 'April 20, 2004'}


In [29]:
roster_table

In [30]:
players

[]

# progran begins


In [55]:
from selenium import webdriver
from bs4 import BeautifulSoup, Comment
from selenium.webdriver.chrome.service import Service

def get_page_source(url: str) -> str:
    # Start Chrome
    driver = webdriver.Chrome()
    driver.get(url)

    html = driver.page_source
    driver.quit()

    return html

def parse_roster_table(html: str, team) -> list:
    soup = BeautifulSoup(html, "html.parser")

    # Find the roster table
    roster_table = soup.find("table", {"id": "roster"})

    if not roster_table:
        raise ValueError("Could not find roster table.")

    # Extract rows from the roster table
    rows = roster_table.find("tbody").find_all("tr")

    players = []
    for row in rows:
        player = {}

        # Player name and href
        player_cell = row.find("td", {"data-stat": "player"})
        player["name"] = player_cell.get_text(strip=True)

        player["team"] = team

        # Jersey number
        player["number"] = row.find("th", {"data-stat": "number"}).get_text(strip=True)

       
        player["href"] = "https://www.basketball-reference.com" + player_cell.find("a")["href"]

        # Birth date
        player["birth_date"] = row.find("td", {"data-stat": "birth_date"}).get_text(strip=True)

        players.append(player)

    return players


def main():
    teams = ["MIA", "LAL", "BOS", "HOU", "DEN"]
    players = []
    for team in teams:
        url = f"https://www.basketball-reference.com/teams/{team}/2026.html"
        html = get_page_source(url)
        players.extend(parse_roster_table(html, team))

    for player in players:
        print(player)


In [56]:
main()

{'name': "Kel'el Ware", 'team': 'MIA', 'number': '7', 'href': 'https://www.basketball-reference.com/players/w/wareke01.html', 'birth_date': 'April 20, 2004'}
{'name': 'Dru Smith', 'team': 'MIA', 'number': '12', 'href': 'https://www.basketball-reference.com/players/s/smithdr01.html', 'birth_date': 'December 30, 1997'}
{'name': 'Jaime Jaquez Jr.', 'team': 'MIA', 'number': '11', 'href': 'https://www.basketball-reference.com/players/j/jaqueja01.html', 'birth_date': 'February 18, 2001'}
{'name': 'Davion Mitchell', 'team': 'MIA', 'number': '45', 'href': 'https://www.basketball-reference.com/players/m/mitchda01.html', 'birth_date': 'September 5, 1998'}
{'name': 'Simone Fontecchio', 'team': 'MIA', 'number': '0', 'href': 'https://www.basketball-reference.com/players/f/fontesi01.html', 'birth_date': 'December 9, 1995'}
{'name': 'Andrew Wiggins', 'team': 'MIA', 'number': '22', 'href': 'https://www.basketball-reference.com/players/w/wiggian01.html', 'birth_date': 'February 23, 1995'}
{'name': 'Pel